In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("Evaluation J3")
         .enableHiveSupport().getOrCreate())


In [2]:
ventes = spark.read.csv("data/j3/ventes_magasins.csv", header = True, inferSchema= True)
ventes.printSchema()
ventes.show()

root
 |-- id_vente: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- magasin: string (nullable = true)
 |-- produit: string (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- vendeur: string (nullable = true)

+--------+----------+--------+--------+--------+-------------+-------+
|id_vente|      date| magasin| produit|quantite|prix_unitaire|vendeur|
+--------+----------+--------+--------+--------+-------------+-------+
|       1|2026-06-01|   Lille| Clavier|       5|         49.9|  Fatou|
|       2|2026-03-11|   Lille| Clavier|       1|         49.9|   NULL|
|       3|2026-01-05|   Paris|  Casque|       1|         79.5|   Chen|
|       4|2026-01-23|   Paris|  Casque|       1|         79.5|  Fatou|
|       5|2026-03-24|  Nantes|  Bureau|       3|        299.0|   NULL|
|       6|2026-02-18|  Nantes|  Chaise|       4|        159.0|   Chen|
|       7|2026-06-05|   Lille|   Écran|    NULL|        189.0|   Éric|
|     

In [3]:
ventes.toPandas().isna().sum()

id_vente          0
date              0
magasin           0
produit           0
quantite         37
prix_unitaire    19
vendeur          58
dtype: int64

In [4]:
ventes.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ventes.columns]).show()


+--------+----+-------+-------+--------+-------------+-------+
|id_vente|date|magasin|produit|quantite|prix_unitaire|vendeur|
+--------+----+-------+-------+--------+-------------+-------+
|       0|   0|      0|      0|      37|           19|     58|
+--------+----+-------+-------+--------+-------------+-------+



In [5]:
ventes_propre = ventes.dropna(subset=["quantite", "prix_unitaire"]).fillna({"vendeur": "inconnu"})

In [6]:
ventes_propre.count(), ventes.count()

(546, 600)

In [7]:
propres = ventes_propre.withColumn("montant", F.col("quantite") * F.col("prix_unitaire"))


In [8]:
propres.show()

+--------+----------+--------+--------+--------+-------------+-------+-------+
|id_vente|      date| magasin| produit|quantite|prix_unitaire|vendeur|montant|
+--------+----------+--------+--------+--------+-------------+-------+-------+
|       1|2026-06-01|   Lille| Clavier|       5|         49.9|  Fatou|  249.5|
|       2|2026-03-11|   Lille| Clavier|       1|         49.9|inconnu|   49.9|
|       3|2026-01-05|   Paris|  Casque|       1|         79.5|   Chen|   79.5|
|       4|2026-01-23|   Paris|  Casque|       1|         79.5|  Fatou|   79.5|
|       5|2026-03-24|  Nantes|  Bureau|       3|        299.0|inconnu|  897.0|
|       6|2026-02-18|  Nantes|  Chaise|       4|        159.0|   Chen|  636.0|
|       8|2026-01-20|   Paris| Clavier|       1|         49.9|   Éric|   49.9|
|       9|2026-03-13|  Nantes|   Écran|       4|        189.0|  Bruno|  756.0|
|      10|2026-02-18|   Paris| Clavier|       2|         49.9|  Fatou|   99.8|
|      11|2026-03-07|  Nantes| Clavier|       5|    

In [9]:
propres.groupBy("magasin").agg(F.count("id_vente").alias("nb_ventes"), 
                               F.round(F.sum("montant"), 2).alias("CA")).orderBy(F.desc("CA")).show()
                               

+--------+---------+-------+
| magasin|nb_ventes|     CA|
+--------+---------+-------+
|  Nantes|      113|53982.7|
|    Lyon|      111|46593.1|
|   Paris|      114|45410.8|
|Bordeaux|      107|44843.3|
|   Lille|      101|43660.3|
+--------+---------+-------+



In [10]:
mag_prod_ca = propres.groupBy("magasin","produit").agg(F.sum("montant").alias("CA_produit"))                                                                               
mag_prod_ca.show()

+--------+--------+------------------+
| magasin| produit|        CA_produit|
+--------+--------+------------------+
|Bordeaux|  Casque|            4054.5|
|   Lille|  Casque|            3736.5|
|    Lyon| Clavier|2944.1000000000004|
|    Lyon|   Écran|            9828.0|
|   Lille| Clavier|            3592.8|
|  Nantes| Clavier|            2894.2|
|  Nantes|  Casque|            1987.5|
|   Lille|  Chaise|            7632.0|
|Bordeaux|  Chaise|            7950.0|
|   Lille|  Bureau|           11362.0|
|Bordeaux|Enceinte|            8772.0|
|    Lyon|  Bureau|           17342.0|
|   Paris|  Chaise|            7155.0|
|  Nantes|  Bureau|           19734.0|
|   Paris|  Bureau|           17641.0|
|   Lille|   Écran|            8694.0|
|   Paris|   Écran|            5859.0|
|Bordeaux| Clavier|2594.7999999999997|
|   Paris|Enceinte|            7095.0|
|Bordeaux|   Écran|            8316.0|
+--------+--------+------------------+
only showing top 20 rows


In [11]:
from pyspark.sql import Window
from pyspark.sql import functions as F


fenetre = Window.partitionBy("magasin").orderBy(F.desc("CA_produit"))
(mag_prod_ca.withColumn("rang", F.row_number().over(fenetre))
      .filter(F.col("rang") <= 2)
      .select("rang", "magasin", "produit", "CA_produit")
      .orderBy("magasin","rang").show())


+----+--------+--------+----------+
|rang| magasin| produit|CA_produit|
+----+--------+--------+----------+
|   1|Bordeaux|  Bureau|   13156.0|
|   2|Bordeaux|Enceinte|    8772.0|
|   1|   Lille|  Bureau|   11362.0|
|   2|   Lille|   Écran|    8694.0|
|   1|    Lyon|  Bureau|   17342.0|
|   2|    Lyon|   Écran|    9828.0|
|   1|  Nantes|  Bureau|   19734.0|
|   2|  Nantes|   Écran|   12663.0|
|   1|   Paris|  Bureau|   17641.0|
|   2|   Paris|  Chaise|    7155.0|
+----+--------+--------+----------+



In [12]:
propres.show()

+--------+----------+--------+--------+--------+-------------+-------+-------+
|id_vente|      date| magasin| produit|quantite|prix_unitaire|vendeur|montant|
+--------+----------+--------+--------+--------+-------------+-------+-------+
|       1|2026-06-01|   Lille| Clavier|       5|         49.9|  Fatou|  249.5|
|       2|2026-03-11|   Lille| Clavier|       1|         49.9|inconnu|   49.9|
|       3|2026-01-05|   Paris|  Casque|       1|         79.5|   Chen|   79.5|
|       4|2026-01-23|   Paris|  Casque|       1|         79.5|  Fatou|   79.5|
|       5|2026-03-24|  Nantes|  Bureau|       3|        299.0|inconnu|  897.0|
|       6|2026-02-18|  Nantes|  Chaise|       4|        159.0|   Chen|  636.0|
|       8|2026-01-20|   Paris| Clavier|       1|         49.9|   Éric|   49.9|
|       9|2026-03-13|  Nantes|   Écran|       4|        189.0|  Bruno|  756.0|
|      10|2026-02-18|   Paris| Clavier|       2|         49.9|  Fatou|   99.8|
|      11|2026-03-07|  Nantes| Clavier|       5|    

In [13]:
propres.write.mode("overwrite").saveAsTable("ventes_propres")

In [14]:
!ls spark-warehouse spark-warehouse/meteo_propre

spark-warehouse:
meteo_propre  ventes_propres

spark-warehouse/meteo_propre:
_SUCCESS  part-00000-56682a6f-1ca5-4661-860d-c75dd8f1351d-c000.snappy.parquet


In [16]:
spark.sql("show tables").show()

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|  default|  meteo_propre|      false|
|  default|ventes_propres|      false|
+---------+--------------+-----------+



In [21]:
spark.sql("""
SELECT VENDEUR, COUNT(*) AS NB_VENTES
FROM ventes_propres
GROUP BY VENDEUR
ORDER BY NB_VENTES DESC, VENDEUR ASC
"""
).show()

+-------+---------+
|VENDEUR|NB_VENTES|
+-------+---------+
|  Amina|       91|
|   Éric|       91|
|   Chen|       84|
|  Fatou|       82|
|  Diane|       79|
|  Bruno|       67|
|inconnu|       52|
+-------+---------+



In [22]:
import json
import os

villes = ["Paris", "Lyon", "Lille", "Nantes", "Bordeaux"]
statuts = ["payee"] * 7 + ["annulee"] * 2 + ["remboursee"]

def envoyer_lot(numero, dossier="data/j3/flux", taille=50):
    """Dépose le lot n° numero : 50 commandes passées dans la n-ième tranche de 5 minutes après 9 h."""
    g = 7919 * numero
    def alea(n):
        nonlocal g
        g = (g * 1103515245 + 12345) % 2**31
        return (g // 65536) % n
    os.makedirs(dossier, exist_ok=True)
    temporaire = f"data/j3/lot_{numero:03d}.tmp"
    with open(temporaire, "w", encoding="utf-8") as f:
        for i in range(taille):
            minute = (numero - 1) * 5 + alea(5)
            evenement = {
                "horodatage": f"2026-10-09 {9 + minute // 60:02d}:{minute % 60:02d}:{alea(60):02d}",
                "id_commande": numero * 1000 + i,
                "ville": villes[alea(5)],
                "montant": (500 + alea(19500)) / 100,
                "statut": statuts[alea(10)],
            }
            f.write(json.dumps(evenement) + "\n")
    os.replace(temporaire, f"{dossier}/lot_{numero:03d}.json")   # le fichier apparaît d'un coup, complet
    print(f"Lot {numero} déposé dans {dossier} : {taille} commandes")

os.makedirs("data/j3/flux_eval", exist_ok=True)   # le dossier que votre flux surveillera


In [24]:
schema = "horodatage STRING, id_commande INT, ville STRING, montant DOUBLE, statut STRING"
flux = spark.readStream.schema(schema).json("data/j3/flux_eval")

In [ ]:
payees = (flux.filter(F.col("statut") == "payee")
              .withColumn("horodatage", F.to_timestamp("horodatage"))
              .select("horodatage", "id_commande", "ville", "montant"))

In [28]:
calcul_commandes = flux.groupBy("statut").agg(F.count("*").alias("nb_commandes"), 
                           F.round(F.sum("montant"), 2).alias("CA"))
                                                     

In [29]:
requete = (calcul_commandes.writeStream.format("memory").queryName("par_statut")
                 .outputMode("complete").start())

In [30]:
for n in (21, 22, 23):
    envoyer_lot(n, dossier="data/j3/flux_eval")

Lot 21 déposé dans data/j3/flux_eval : 50 commandes
Lot 22 déposé dans data/j3/flux_eval : 50 commandes
Lot 23 déposé dans data/j3/flux_eval : 50 commandes


In [31]:
requete.processAllAvailable()

In [34]:
spark.sql("SELECT * FROM par_statut").show()

+----------+------------+-------+
|    statut|nb_commandes|     CA|
+----------+------------+-------+
|     payee|         100|8352.14|
|remboursee|          17|1635.51|
|   annulee|          33|2998.47|
+----------+------------+-------+



In [35]:
envoyer_lot(24, dossier="data/j3/flux_eval")

Lot 24 déposé dans data/j3/flux_eval : 50 commandes


In [36]:
requete.processAllAvailable()

In [37]:
spark.sql("SELECT * FROM par_statut").show()

+----------+------------+--------+
|    statut|nb_commandes|      CA|
+----------+------------+--------+
|     payee|         140|12237.71|
|remboursee|          22| 2261.44|
|   annulee|          38| 3334.29|
+----------+------------+--------+



In [38]:
requete.stop()